In [91]:
# Import Libraries
import pandas as pd
import numpy as np
import us
from sklearn.preprocessing import StandardScaler

# Cleaning the NORS Outbreak Dataset

In [92]:
# Load the dataset
outbreak_df = pd.read_csv("NORS_outbreaks.csv", low_memory = False)

# View the data
outbreak_df.head()

,Year,Month,State,Primary Mode,Etiology,Serotype or Genotype,Etiology Status,Setting,Illnesses,Hospitalizations,Info On Hospitalizations,Deaths,Info On Deaths,Food Vehicle,Food Contaminated Ingredient,IFSAC Category,Water Exposure,Water Type,Animal Type
0,1971,2,California,Water,Copper,NaN,Confirmed,Restaurant,2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN
1,1971,6,Arkansas,Water,Hepatitis A,NaN,Confirmed,Store,98,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Other,NaN
2,1971,6,Missouri,Water,Unknown,NaN,Suspected,Subdivision/Neighborhood,2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN
3,1971,6,Alabama,Water,Selenium,NaN,Confirmed,Unknown,3,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Individual/Private,NaN
4,1971,6,Vermont,Water,Unknown,NaN,Suspected,Community/municipality,3,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN


In [93]:
# Make a copy of the original dataset to clean
outbreak_clean = outbreak_df.copy()

# Rename all columns to lowercase and replace spaces with underscores
outbreak_clean.columns = outbreak_clean.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^0-9a-zA-Z_]', '', regex=True)

## Understand/Inspecting the Dataset

- Identify all columns (e.g., outbreak ID, state, illness type, number of cases, dates, settings).
- Check for categorical vs numeric variables.

In [94]:
# Show general info (types, missing values)
print("\nDataset Info:")
print(outbreak_clean.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66713 entries, 0 to 66712
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year                          66713 non-null  int64  
 1   month                         66713 non-null  int64  
 2   state                         66713 non-null  object 
 3   primary_mode                  66713 non-null  object 
 4   etiology                      50375 non-null  object 
 5   serotype_or_genotype          16470 non-null  object 
 6   etiology_status               50375 non-null  object 
 7   setting                       60804 non-null  object 
 8   illnesses                     66713 non-null  object 
 9   hospitalizations              58155 non-null  float64
 10  info_on_hospitalizations      58480 non-null  object 
 11  deaths                        58785 non-null  float64
 12  info_on_deaths                58463 non-null 

In [95]:
# Show summary statistics for numeric columns
print("\nSummary Statistics:")
display(outbreak_clean.describe())


Summary Statistics:


,year,month,hospitalizations,deaths
count,66713.000000,66713.000000,58155.000000,58785.000000
mean,2012.696926,5.692009,0.825501,0.042375
std,7.546412,3.736764,4.164555,0.436484
min,1971.000000,1.000000,0.000000,0.000000
25%,2009.000000,2.000000,0.000000,0.000000
50%,2014.000000,5.000000,0.000000,0.000000
75%,2018.000000,9.000000,1.000000,0.000000
max,2023.000000,12.000000,308.000000,50.000000


**Determine unique values of categorical variables**

**1.  States**

In [96]:
categorical_columns = ['state']
for col in categorical_columns:
    if col in outbreak_clean.columns:
        print(f"\nUnique values in {col}:")
        print(outbreak_clean[col].unique())


Unique values in state:
['California' 'Arkansas' 'Missouri' 'Alabama' 'Vermont' 'Oregon'
 'New Jersey' 'Mississippi' 'Kentucky' 'Oklahoma' 'New Mexico'
 'North Carolina' 'New York' 'Alaska' 'Texas' 'Indiana' 'Colorado' 'Ohio'
 'Minnesota' 'Illinois' 'Florida' 'Pennsylvania' 'Washington' 'Maryland'
 'Massachusetts' 'Tennessee' 'Utah' 'Hawaii' 'Iowa' 'West Virginia'
 'Arizona' 'Virginia' 'Connecticut' 'Idaho' 'New Hampshire' 'Wisconsin'
 'Montana' 'Puerto Rico' 'Louisiana' 'Kansas' 'South Carolina' 'Maine'
 'Wyoming' 'North Dakota' 'Michigan' 'Georgia' 'South Dakota'
 'Rhode Island' 'Nevada' 'Virgin Islands' 'Multistate' 'Delaware'
 'Northern Mariana Islands' 'Nebraska' 'Guam' 'District of Columbia'
 'Republic of the Marshall Islands' 'Republic of Palau']


**2. Primary Mode**

In [97]:
categorical_columns = ['primary_mode']
for col in categorical_columns:
    if col in outbreak_clean.columns:
        print(f"\nUnique values in {col}:")
        print(outbreak_clean[col].unique())


Unique values in primary_mode:
['Water' 'Food' 'Person-to-person' 'Indeterminate/unknown'
 'Animal contact' 'Environmental contamination other than food/water']


**3. Etiology**

In [98]:
# Create a new grouped column for etiology
conditions = [
    outbreak_clean['etiology'].str.contains('norovirus', case=False, na=False),
    outbreak_clean['etiology'].str.contains('salmonella', case=False, na=False),
    outbreak_clean['etiology'].str.contains('campylobacter', case=False, na=False),
    outbreak_clean['etiology'].str.contains('clostridium', case=False, na=False),
    outbreak_clean['etiology'].str.contains('staphylococcus', case=False, na=False),
    outbreak_clean['etiology'].str.contains('hepatitis', case=False, na=False),
    outbreak_clean['etiology'].str.contains('unknown|unspecified|not determined', case=False, na=False)
]

choices = [
    'Norovirus',
    'Salmonella',
    'Campylobacter',
    'Clostridium',
    'Staphylococcus',
    'Hepatitis',
    'Unknown'
]

# Create new column
outbreak_clean['etiology_grouped'] = np.select(conditions, choices, default='Other')

# Check unique values
outbreak_clean['etiology_grouped'].value_counts()

etiology_grouped
Norovirus         33157
Other             23935
Salmonella         4582
Clostridium        1575
Unknown            1451
Campylobacter      1109
Staphylococcus      757
Hepatitis           147
Name: count, dtype: int64

**4. Serotype or Genotype**

In [99]:
outbreak_clean['serotype_or_genotype'].nunique()

765

We'll keep this variable as-is since it describes in more detail the etiology of the outbreak

**5. Etiology Status**

In [100]:
outbreak_clean['etiology_status'].nunique()

57

In [101]:
categorical_columns = ['etiology_status']
for col in categorical_columns:
    if col in outbreak_clean.columns:
        print(f"\nUnique values in {col}:")
        print(outbreak_clean[col].unique())


Unique values in etiology_status:
['Confirmed' 'Suspected' nan 'Suspected;Confirmed' 'Confirmed;Confirmed'
 'Confirmed;Confirmed;Suspected;Suspected' 'Confirmed;Suspected'
 'Confirmed;Confirmed;Confirmed' 'Suspected;Suspected;Confirmed'
 'Confirmed;Confirmed;Confirmed;Confirmed'
 'Suspected;Suspected;Confirmed;Suspected' 'Suspected;Suspected'
 'Suspected;Suspected;Confirmed;Confirmed'
 'Suspected;Suspected;Suspected;Confirmed' 'Suspected;Confirmed;Suspected'
 'Suspected;Suspected;Suspected;Suspected' 'Confirmed;Confirmed;Suspected'
 'Suspected;Suspected;Suspected'
 'Confirmed;Confirmed;Confirmed;Confirmed;Confirmed;Confirmed'
 'Confirmed;Confirmed;Confirmed;Confirmed;Confirmed'
 'Confirmed;Suspected;Confirmed;Confirmed' 'Suspected;Confirmed;Confirmed'
 'Confirmed;Confirmed;Confirmed;Confirmed;Suspected'
 'Suspected;Confirmed;Confirmed;Confirmed;Confirmed;Confirmed;Confirmed'
 'Confirmed;Suspected;Confirmed' 'Confirmed;Suspected;Suspected'
 'Confirmed;Suspected;Suspected;Confirmed'
 'S

Very messy, possibly omit this variable

**6. Setting**

In [102]:
outbreak_clean['setting'].nunique()

597

Many different settings, some have similarities, might be worth exploring ways to group them.

**7. Food Vehicle**

In [103]:
# Top 10 food vehicles by frequency
top_food_vehicles = (
    outbreak_clean['food_vehicle']
    .value_counts()
    .head(10)
    .reset_index()
    .rename(columns={'index': 'food_vehicle', 'food_vehicle': 'count'})
)

top_food_vehicles

,count,count
0,"oysters, raw",264
1,multiple foods,208
2,"ground beef, hamburger",134
3,"salad, unspecified",130
4,chicken,115
5,"chicken, unspecified",104
6,"sandwich, submarine",92
7,"pork, BBQ",87
8,"chicken, other",85
9,"fish, mahi mahi",85


In [104]:
# Investigate missing food_vehicle values by parimary_mode

# Mark missing vs present food_vehicle values
outbreak_clean['food_vehicle_missing'] = outbreak_clean['food_vehicle'].apply(
    lambda x: "Missing" if pd.isna(x) or str(x).strip() == "" else "Present"
)

# Count occurrences and compute percentages by primary_mode
food_missing_summary = (
    outbreak_clean
    .groupby(['primary_mode', 'food_vehicle_missing'])
    .size()
    .reset_index(name='count')
)

food_missing_summary['percent'] = (
    food_missing_summary.groupby('primary_mode')['count']
    .transform(lambda x: 100 * x / x.sum())
)

food_missing_summary

,primary_mode,food_vehicle_missing,count,percent
0,Animal contact,Missing,655,100.000000
1,Environmental contamination other than food/water,Missing,132,100.000000
2,Food,Missing,12283,49.670427
3,Food,Present,12446,50.329573
4,Indeterminate/unknown,Missing,5677,100.000000
5,Person-to-person,Missing,32414,100.000000
6,Water,Missing,3106,100.000000


**8. Food Contaminated Ingredients**

In [105]:
outbreak_clean['food_contaminated_ingredient'].nunique()

518

This might be a redundant variable, consider omitting and just using food vehicle for sake of simplicity.

**3.  IFSAC Category**

IFSAC stands for the Interagency Food Safety Analytics Collaboration, which is a categorization scheme used to identify foods most often linked to specific illnesses caused by certain pathogens. Source: <https://www.cdc.gov/ifsac/php/projects/food-categorization-scheme.html#:~:text=At%20a%20glance,include%20ice%20and%20dietary%20supplements.>

In [106]:
outbreak_clean['ifsac_category'].nunique()

25

This category is similar to the food vehicle category; because it's more standardized, it might be worth using instead of food category.

**9. Water Exposure**

In [107]:
# Top water exposure types by frequency
water_exposure_counts = (
    outbreak_clean['water_exposure']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'water_exposure', 'water_exposure': 'count'})
)

water_exposure_counts.head(10)

,count,count
0,NaN,63607
1,Drinking water,1256
2,Recreational water -- treated,1085
3,Recreational water -- untreated,361
4,Undetermined water,192
5,Other/Environmental water,180
6,Drinking water;Recreational water -- treated,14
7,Drinking water;Other/Environmental water,11
8,Drinking water;Undetermined water,5
9,Other/Environmental water;Recreational water -...,1


In [108]:
# Create a column flagging missing vs present values
outbreak_clean['water_exposure_missing'] = outbreak_clean['water_exposure'].apply(
    lambda x: "Missing" if pd.isna(x) or str(x).strip() == "" else "Present"
)

# Count by primary_mode and missing/present status
water_missing_summary = (
    outbreak_clean
    .groupby(['primary_mode', 'water_exposure_missing'])
    .size()
    .reset_index(name='count')
)

# Calculate percentages within each primary mode
water_missing_summary['percent'] = (
    water_missing_summary.groupby('primary_mode')['count']
    .transform(lambda x: 100 * x / x.sum())
)

water_missing_summary

,primary_mode,water_exposure_missing,count,percent
0,Animal contact,Missing,655,100.0
1,Environmental contamination other than food/water,Missing,132,100.0
2,Food,Missing,24729,100.0
3,Indeterminate/unknown,Missing,5677,100.0
4,Person-to-person,Missing,32414,100.0
5,Water,Present,3106,100.0


All outbreaks within water as primary mode of exposure have a designated type of water exposure (in other words, no missing values of water exposure in water-borne outbreaks)

**10. Water Type**

In [109]:
# Number of unique water types
outbreak_clean['water_type'].nunique()

56

In [110]:
# Top 10 water types
top_water_types = (
    outbreak_clean['water_type']
    .value_counts(dropna=False)
    .head(10)
    .reset_index()
    .rename(columns={'index': 'water_type', 'water_type': 'count'})
)

top_water_types

,count,count
0,NaN,63983
1,Community,693
2,Pool - Other Swimming Pool,511
3,Hot Tub/Spa/Whirlpool,348
4,Other,346
5,Lake/Reservoir,283
6,Hot Tub/Spa/Whirlpool;Pool - Other Swimming Pool,113
7,Individual/Private,112
8,Unknown,76
9,Splash Pad/Interactive Fountain /Water Playground,41


In [111]:
# Group water types into broader categories
# Conditions for grouping
conditions = [
    outbreak_clean['water_type'].isna(),
    outbreak_clean['water_type'].str.contains('community|individual|private', case=False, na=False),
    outbreak_clean['water_type'].str.contains('pool|hot tub|spa|whirlpool|splash pad|fountain|water playground', case=False, na=False),
    outbreak_clean['water_type'].str.contains('lake|reservoir|other', case=False, na=False),
    outbreak_clean['water_type'].str.contains('unknown|unspecified|not determined', case=False, na=False)
]

choices = [
    'Missing/NA',
    'Drinking/Community Water',
    'Recreational Water',
    'Natural/Other',
    'Unknown'
]

# Create new grouped column
outbreak_clean['water_type_group'] = np.select(conditions, choices, default='Other')

# Check counts of the new grouping
outbreak_clean['water_type_group'].value_counts()

water_type_group
Missing/NA                  63983
Recreational Water           1060
Drinking/Community Water      856
Natural/Other                 646
Other                          92
Unknown                        76
Name: count, dtype: int64

**11. Animal Type**

In [112]:
# Top 10 animal types
top_animal_types = (
    outbreak_clean['animal_type']
    .value_counts(dropna=False)
    .head(10)
    .reset_index()
    .rename(columns={'index': 'animal_type', 'animal_type': 'count'})
)

top_animal_types

,count,count
0,NaN,66126
1,Poultry,184
2,Cattle,147
3,Dog or puppy,46
4,Turtle,41
5,Goat,30
6,Lizard,25
7,Pig,8
8,Hedgehog,7
9,Sheep,7


In [113]:
# Group animal types into broader categories
# Conditions for grouping
conditions = [
    outbreak_clean['animal_type'].isna(),
    outbreak_clean['animal_type'].str.contains('poultry', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('cattle', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('cat|dog|ferret|guinea pig|hedgehog|mouse|rat|raccoon', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('horse|pony|goat|sheep|pig|alpaca|donkey|llama|yak', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('lizard|turtle|snake|reptile', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('bird, not including poultry', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('fish', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('frog|salamander', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('other', case=False, na=False)
]

choices = [
    'Missing/NA',
    'Poultry',
    'Cattle',
    'House Pets',
    'Other Ungulates',
    'Reptiles',
    'Other Birds',
    'Fish',
    'Amphibians',
    'Other/Unknown'
]

# Create new grouped column
outbreak_clean['animal_group'] = np.select(conditions, choices, default='Other/Unknown')

# Check counts of the new grouping
outbreak_clean['animal_group'].value_counts()

animal_group
Missing/NA         66126
Poultry              209
Cattle               166
Reptiles              75
House Pets            70
Other Ungulates       56
Other/Unknown          6
Fish                   3
Amphibians             2
Name: count, dtype: int64

In [114]:
# View the clean dataset
outbreak_clean.head()

,year,month,state,primary_mode,etiology,serotype_or_genotype,etiology_status,setting,illnesses,hospitalizations,...,food_contaminated_ingredient,ifsac_category,water_exposure,water_type,animal_type,etiology_grouped,food_vehicle_missing,water_exposure_missing,water_type_group,animal_group
0,1971,2,California,Water,Copper,NaN,Confirmed,Restaurant,2,NaN,...,NaN,NaN,Drinking water,Community,NaN,Other,Missing,Present,Drinking/Community Water,Missing/NA
1,1971,6,Arkansas,Water,Hepatitis A,NaN,Confirmed,Store,98,NaN,...,NaN,NaN,Drinking water,Other,NaN,Hepatitis,Missing,Present,Natural/Other,Missing/NA
2,1971,6,Missouri,Water,Unknown,NaN,Suspected,Subdivision/Neighborhood,2,NaN,...,NaN,NaN,Drinking water,Community,NaN,Unknown,Missing,Present,Drinking/Community Water,Missing/NA
3,1971,6,Alabama,Water,Selenium,NaN,Confirmed,Unknown,3,NaN,...,NaN,NaN,Drinking water,Individual/Private,NaN,Other,Missing,Present,Drinking/Community Water,Missing/NA
4,1971,6,Vermont,Water,Unknown,NaN,Suspected,Community/municipality,3,NaN,...,NaN,NaN,Drinking water,Community,NaN,Unknown,Missing,Present,Drinking/Community Water,Missing/NA


## Handle Missing Data

- Remove columns with mostly missing values.
- Impute missing values if necessary (mean/mode or “Unknown”).

In [115]:
# Drop all unnecessary variables
columns_to_drop = [
    'info_on_hospitalizations',
    'info_on_deaths',
    'etiology_status',
    'food_vehicle',
    'food_contaminated_ingredient'
]

outbreak_clean = outbreak_clean.drop(columns=columns_to_drop)

In [116]:
# Update missing values with correct labeling
# Define categorical columns (general)
categorical_cols = ['etiology', 'setting']

# Define contingent categorical columns
contingent_cols = [
    'serotype_or_genotype','ifsac_category', 'water_exposure', 'water_type', 'animal_type'
]

# Fill missing values in categorical columns with 'Unknown'
for col in categorical_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna('Unknown')

# Fill missing values in contingent columns with 'Not Applicable'
for col in contingent_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna('Not Applicable')

# Handle numeric columns
numeric_cols = ['hospitalizations', 'deaths']
for col in numeric_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna(outbreak_clean[col].median())

# Check that all missing values are handled
print("Missing values after cleaning:")
print(outbreak_clean.isnull().sum())

Missing values after cleaning:
year                      0
month                     0
state                     0
primary_mode              0
etiology                  0
serotype_or_genotype      0
setting                   0
illnesses                 0
hospitalizations          0
deaths                    0
ifsac_category            0
water_exposure            0
water_type                0
animal_type               0
etiology_grouped          0
food_vehicle_missing      0
water_exposure_missing    0
water_type_group          0
animal_group              0
dtype: int64


## Standardize and Normalize Data

- Standardize categorical variables (e.g., state abbreviations, illness names).
- Normalize numeric variables if you plan to use clustering or distance-based methods.

In [117]:
# Create a 'date' column from 'year' and 'month'
outbreak_clean['date'] = pd.to_datetime(outbreak_clean[['year', 'month']].assign(day=1))

# Add a new column for state abbreviations
def get_state_abbr(state):
    state_info = us.states.lookup(state)
    return state_info.abbr if state_info else 'Unknown'

outbreak_clean['state_abbr'] = outbreak_clean['state'].apply(get_state_abbr)

In [118]:
# Standardize numeric values using z-score
# Select numeric columns to standardize
numeric_cols = ['hospitalizations', 'deaths']

# Initialize scaler
scaler = StandardScaler()

# Fit and transform numeric columns
outbreak_clean[numeric_cols] = scaler.fit_transform(outbreak_clean[numeric_cols])

# Preview the standardized numeric columns
display(outbreak_clean[numeric_cols].head())

,hospitalizations,deaths
0,-0.184607,-0.091081
1,-0.184607,-0.091081
2,-0.184607,-0.091081
3,-0.184607,-0.091081
4,-0.184607,-0.091081


## Feature Engineering
1. Outbreak Severity
Combine illnesses, hospitalizations, and deaths into a single severity measure. Since illnesses might still be an object (string), we first convert it to numeric if needed.

In [119]:
# Ensure illnesses column is numeric
outbreak_clean['illnesses'] = pd.to_numeric(outbreak_clean['illnesses'], errors='coerce').fillna(0)

# Create outbreak_severity as the sum of illnesses, hospitalizations, and deaths
outbreak_clean['outbreak_severity'] = outbreak_clean['illnesses'] + outbreak_clean['hospitalizations'] + outbreak_clean['deaths']

# Preview
display(outbreak_clean[['illnesses', 'hospitalizations', 'deaths', 'outbreak_severity']].head())

,illnesses,hospitalizations,deaths,outbreak_severity
0,2.0,-0.184607,-0.091081,1.724312
1,98.0,-0.184607,-0.091081,97.724312
2,2.0,-0.184607,-0.091081,1.724312
3,3.0,-0.184607,-0.091081,2.724312
4,3.0,-0.184607,-0.091081,2.724312


2. Binary Features Based on Primary Mode
Create flags for food, water, or animal-related outbreaks:

In [120]:
outbreak_clean['is_food_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Food' in x else 0)
outbreak_clean['is_water_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Water' in x else 0)
outbreak_clean['is_animal_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Animal' in x else 0)

# Preview
display(outbreak_clean[['primary_mode', 'is_food_related', 'is_water_related', 'is_animal_related']].head())

,primary_mode,is_food_related,is_water_related,is_animal_related
0,Water,0,1,0
1,Water,0,1,0
2,Water,0,1,0
3,Water,0,1,0
4,Water,0,1,0


3. Outbreak Month/Season
You can categorize outbreaks into seasons to see if there’s seasonality:

In [121]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

outbreak_clean['season'] = outbreak_clean['month'].apply(month_to_season)

# Preview
display(outbreak_clean[['month', 'season']].head())

,month,season
0,2,Winter
1,6,Summer
2,6,Summer
3,6,Summer
4,6,Summer


In [123]:
# Reorder the columns
outbreak_clean = outbreak_clean[[col for col in [
    'date', 'year', 'month', 'season', 'state', 'state_abbr',
    'primary_mode', 'etiology', 'serotype_or_genotype', 'ifsac_category', 'setting',
    'illnesses', 'hospitalizations', 'deaths', 'outbreak_severity',
    'food_vehicle_missing', 'water_exposure', 'water_exposure_missing', 'water_type', 'water_type_group',
    'animal_type', 'animal_group',
    'is_food_related', 'is_water_related', 'is_animal_related'
] if col in outbreak_clean.columns]]

# Preview cleaned dataframe
display(outbreak_clean.head())

,date,year,month,season,state,state_abbr,primary_mode,etiology,serotype_or_genotype,ifsac_category,...,food_vehicle_missing,water_exposure,water_exposure_missing,water_type,water_type_group,animal_type,animal_group,is_food_related,is_water_related,is_animal_related
0,1971-02-01,1971,2,Winter,California,CA,Water,Copper,Not Applicable,Not Applicable,...,Missing,Drinking water,Present,Community,Drinking/Community Water,Not Applicable,Missing/NA,0,1,0
1,1971-06-01,1971,6,Summer,Arkansas,AR,Water,Hepatitis A,Not Applicable,Not Applicable,...,Missing,Drinking water,Present,Other,Natural/Other,Not Applicable,Missing/NA,0,1,0
2,1971-06-01,1971,6,Summer,Missouri,MO,Water,Unknown,Not Applicable,Not Applicable,...,Missing,Drinking water,Present,Community,Drinking/Community Water,Not Applicable,Missing/NA,0,1,0
3,1971-06-01,1971,6,Summer,Alabama,AL,Water,Selenium,Not Applicable,Not Applicable,...,Missing,Drinking water,Present,Individual/Private,Drinking/Community Water,Not Applicable,Missing/NA,0,1,0
4,1971-06-01,1971,6,Summer,Vermont,VT,Water,Unknown,Not Applicable,Not Applicable,...,Missing,Drinking water,Present,Community,Drinking/Community Water,Not Applicable,Missing/NA,0,1,0


## Save dataset to CSV

In [124]:
# Save cleaned dataset to CSV
outbreak_clean.to_csv('outbreak_clean_v3.csv', index=False)